# Declare a new run

Bare-minimum template for starting a new run: define its sources, its
tokenizer (reused if one already matches), a dataset and a pretraining config
under a fresh `run_id`, then check and declare it against the volume.

Copy this notebook per run and change `RUN_ID` and the knobs below.
Everything past declaring -- running jobs, loading artifacts back,
visualizing a plan, conflicts -- is `demo.ipynb`, not repeated here.

In [1]:
from dag.artifact import Resources
from datasets.artifact import DataSet
from mappeddatasets.artifact import MappedDataSet
from models.mock.artifact import ModelParameters, Pretraining, PretrainingConfig
from sources.artifact import Source
from tokenizers.bpe import Tokenizer as BPETokenizer

## Sources

In [2]:
odyssey = Source(
    name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt"
)
mobydick = Source(
    name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt"
)
romeojuliet = Source(
    name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt"
)
montecristo = Source(
    name="montecristo", url="https://www.gutenberg.org/cache/epub/1184/pg1184.txt"
)
pride = Source(
    name="pride", url="https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
)
frankenstein = Source(
    name="frankenstein", url="https://www.gutenberg.org/cache/epub/84/pg84.txt"
)
greatexpectations = Source(
    name="greatexpectations", url="https://www.gutenberg.org/cache/epub/1400/pg1400.txt"
)
dracula = Source(
    name="dracula", url="https://www.gutenberg.org/cache/epub/345/pg345.txt"
)


## The run

In [16]:
RUN_ID = "RUN_V7"  # <- change this per run

# same tokenizer any other run trained on these sources would build -- if one's
# already declared, it's reused rather than rebuilt from scratch
tokenizer = BPETokenizer(
    vocab_size=1420,
    special_tokens=("<pad>", "<unk>"),
    sources=(odyssey, mobydick, dracula),
)

# also shared, same as the tokenizer above -- another run asking for
# this tokenizer and these exact sources reuses this train.bin/valid.bin
dataset = DataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[montecristo, frankenstein],
    valid_sources=[pride,odyssey],
)

mappedset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[montecristo, frankenstein],
    valid_sources=[dracula,pride],
)

model_parameters = ModelParameters(hidden_size=64, num_layers=2)
config = PretrainingConfig(
    total_steps=6000, batch_size=32, lr=1e-3, seed=1, checkpoint_every=500
)

pretraining = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset,
    tokenizer=tokenizer,
    model_parameters=model_parameters,
    config=config,
    allocated_resources=Resources(gpu_type='T4', gpu_count=1)
)

pretraining  # parameters all the way down; `commit` is hidden from the repr

Pretraining(run_id='RUN_V7', dataset=MappedDataSet(train_set=(TokenizedSource(tokenizer=Tokenizer(vocab_size=1420, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt'), Source(name='mobydick', url='https://www.gutenberg.org/cache/epub/2701/pg2701.txt'), Source(name='dracula', url='https://www.gutenberg.org/cache/epub/345/pg345.txt'))), source=Source(name='montecristo', url='https://www.gutenberg.org/cache/epub/1184/pg1184.txt')), TokenizedSource(tokenizer=Tokenizer(vocab_size=1420, special_tokens=('<pad>', '<unk>'), sources=(Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt'), Source(name='mobydick', url='https://www.gutenberg.org/cache/epub/2701/pg2701.txt'), Source(name='dracula', url='https://www.gutenberg.org/cache/epub/345/pg345.txt'))), source=Source(name='frankenstein', url='https://www.gutenberg.org/cache/epub/84/pg84.txt'))), valid_set=(TokenizedSource(tokenizer=Tokenizer(v

## Declare

`declare_fn.remote(pretraining)` resolves the request and reconciles it
against the volume without writing anything -- everything shared (sources,
and the tokenizer if it matches one already declared) should read `done`;
everything new to this run should read `new`. `write=True` declares
whatever's `new`, refusing outright if anything's inconsistent.

In [17]:
import modal

from config import APP_NAME

declare_fn = modal.Function.from_name(APP_NAME, "declare")

In [18]:
print(declare_fn.remote(pretraining))  # preview, no writes

run RUN_V7 under /storage
  done       sources/odyssey
  done       sources/mobydick
  done       sources/dracula
  declared   tokenizers/bpe-1.4k-85db95152c
  new        sources/montecristo
  new        tokenizers/bpe-1.4k-85db95152c/bin/montecristo
  done       sources/frankenstein
  declared   tokenizers/bpe-1.4k-85db95152c/bin/frankenstein
  declared   tokenizers/bpe-1.4k-85db95152c/bin/dracula
  done       sources/pride
  declared   tokenizers/bpe-1.4k-85db95152c/bin/pride
  new        mappeddatasets/mapped-c4bf62c829
  new        runs/RUN_V7/pretraining

4 declared, 5 done, 4 new
ok -- 4 to declare


In [22]:
print(declare_fn.remote(pretraining, write=True))  # commits to the volume

run RUN_V7 under /storage
  done       sources/odyssey
  done       sources/mobydick
  done       sources/dracula
  declared   tokenizers/bpe-1.4k-85db95152c
  declared   sources/montecristo
  declared   tokenizers/bpe-1.4k-85db95152c/bin/montecristo
  done       sources/frankenstein
  declared   tokenizers/bpe-1.4k-85db95152c/bin/frankenstein
  declared   tokenizers/bpe-1.4k-85db95152c/bin/dracula
  done       sources/pride
  declared   tokenizers/bpe-1.4k-85db95152c/bin/pride
  declared   mappeddatasets/mapped-c4bf62c829
  declared   runs/RUN_V7/pretraining

8 declared, 5 done
ok -- 0 to declare


Declared, not yet produced -- `declared` rows have a manifest but no files
yet. To actually run the jobs, see `demo.ipynb`'s job-list and `Job.run()`
cells, or drive `job_list(resolve(pretraining))` by hand.

In [ ]:
pretraining.manifest()

In [ ]:
.

## A shared, virtual alternative: `MappedDataSet`

`dataset` above is run-scoped and physically copies every source's tokens into
`train.bin`/`valid.bin` under this run's own folder. `MappedDataSet` is the alternative: no
`run_id`, no copy, no files of its own at all -- its identity is a digest over the
tokenizer and source lists, so it declares once under its own shared `mappeddatasets/`
folder (holding nothing but its manifest) and any run asking for the same tokenizer and
sources reuses it. It's *done* the moment its sources are -- there's no job output of its
own to wait on -- and it reads directly out of each source's own `tokens.bin` through
`mappeddatasets.tokenstream.TokenStream` at bind time (see
[dataset_demo.ipynb](dataset_demo.ipynb) for the full local walkthrough, including why a
window is never allowed to cross from one source into the next).

Not wired into `pretraining` above -- that still takes the physical `dataset` -- this just
declares it standalone, the same way `pretraining` is declared.

In [ ]:
from mappeddatasets.artifact import MappedDataSet

mapped_dataset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[dracula, frankenstein],
    valid_sources=[pride, odyssey],
)

print(mapped_dataset.uid)            # no RUN_ID anywhere in this -- shared across runs
print(mapped_dataset.artifact_path)

In [ ]:
print(declare_fn.remote(mapped_dataset))  # preview, no writes

In [ ]:
print(declare_fn.remote(mapped_dataset, write=True))  # commits to the volume